# Alternative Optimizers

Two experiments on target Ω₂ = {11, 11.2, 13}:
1. **CMA-ES** — derivative-free global search with 20× evaluation budget
2. **Random uniform initialization** — prefactors drawn from Uniform([1, 9])
   compared against standard unary perturbation {1.01, 1.02, 1.03}

Produces data for `fig:cmaes_analysis` and the baseline comparison table.

**Requires:** `pip install cma`

In [ ]:
# ── Smoke test flag ────────────────────────────────────────────
# Set True to run a fast end-to-end check (1 function, 2 seeds, 50 evals)
SMOKE_TEST = False

from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import pennylane as qml
import cma
import pandas as pd
import pickle
import sys
from sklearn.metrics import r2_score

sys.path.append("..")
from paper_style import apply_paper_style, BLUE, RED, GREEN, ORANGE, PURPLE
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

jax.config.update("jax_enable_x64", True)
apply_paper_style()

SEED = 42
np.random.seed(SEED)

## Hyperparameters

In [ ]:
# Smoke test overrides applied below
NUM_FUNCTIONS       = 1 if SMOKE_TEST else 10
NUM_RUNS            = 2 if SMOKE_TEST else 10
CMAES_MAX_FEVALS    = 500 if SMOKE_TEST else 100_000

NUM_WIRES           = 3
NUM_SERIAL_LAYERS   = 2
TRAINABLE_BLOCKS    = 1
NUM_SU_GATES        = 1
NUM_ROT_PARAMS      = 63   # SU(8)
ALPHA_MAX           = 9.0  # upper bound for random uniform init
SIGMA0              = 0.5  # CMA-ES initial step size

DATA_DIR  = Path("../datasets")
OUT_DIR   = Path("results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"SMOKE_TEST={SMOKE_TEST}  functions={NUM_FUNCTIONS}  "
      f"runs={NUM_RUNS}  cmaes_evals={CMAES_MAX_FEVALS}")

## Circuit definition

In [ ]:
def S(alpha, x, num_wires, enc_layer):
    for w in range(num_wires):
        qml.RX(alpha[w] * x, wires=w)

def W_SU(theta, trainable_block_layers, num_wires):
    for i in range(trainable_block_layers):
        qml.SpecialUnitary(theta[i][0], wires=range(num_wires))

def weights_ones(n_serial, t_blocks, n_su, n_params):
    return np.ones((n_serial, t_blocks, n_su, n_params))

def square_loss(targets, predictions):
    return 0.5 * sum((t-p)**2 for t,p in zip(targets, predictions)) / len(targets)

dev = qml.device("default.qubit", wires=NUM_WIRES)

@qml.qnode(dev)
def circuit(alpha, weights_SU, x=None):
    W_SU(weights_SU[0], TRAINABLE_BLOCKS, NUM_WIRES)
    for i in range(NUM_SERIAL_LAYERS - 1):
        S(alpha[i], x, NUM_WIRES, i)
        W_SU(weights_SU[i+1], TRAINABLE_BLOCKS, NUM_WIRES)
    return qml.expval(qml.PauliZ(wires=NUM_WIRES-1))

circuit_jit = jax.jit(circuit)

## Load data

In [ ]:
with open(DATA_DIR / "datasets_1d_jaderberg_10.pkl", "rb") as f:
    datasets = pickle.load(f)
print(f"Loaded {len(datasets)} target functions (Ω₂ = {{11, 11.2, 13}})")

## Experiment 1: CMA-ES

In [ ]:
def cost_numpy(flat_params, x_train, y_train):
    """Flat parameter vector → scalar cost (numpy, for CMA-ES)."""
    n_su  = NUM_SERIAL_LAYERS * TRAINABLE_BLOCKS * NUM_SU_GATES * NUM_ROT_PARAMS
    n_al  = 1 * NUM_WIRES
    weights = jnp.array(flat_params[:n_su].reshape(
        NUM_SERIAL_LAYERS, TRAINABLE_BLOCKS, NUM_SU_GATES, NUM_ROT_PARAMS
    ))
    alpha = jnp.array(flat_params[n_su:n_su+n_al].reshape(1, NUM_WIRES))
    preds = [float(circuit_jit(alpha, weights, x_)) for x_ in x_train]
    return float(square_loss(y_train, preds))

cmaes_results = []

for func in range(NUM_FUNCTIONS):
    ds = datasets[func]
    x_tr, y_tr = ds["x_train"], ds["y_train"]
    x_te, y_te = ds["x_test"],  ds["y_test"]

    for run in range(NUM_RUNS):
        print(f"CMA-ES  func={func+1}/{NUM_FUNCTIONS}  run={run+1}/{NUM_RUNS}")

        # Flat init: ansatz ones + unary alpha {1.01,1.02,1.03}
        w0 = weights_ones(NUM_SERIAL_LAYERS, TRAINABLE_BLOCKS,
                          NUM_SU_GATES, NUM_ROT_PARAMS).ravel()
        a0 = np.array([1.01, 1.02, 1.03])
        x0 = np.concatenate([w0, a0])

        es = cma.CMAEvolutionStrategy(
            x0, SIGMA0,
            {"maxfevals": CMAES_MAX_FEVALS,
             "verbose":   -9,
             "seed":      SEED + run}
        )
        es.optimize(cost_numpy, args=(x_tr, y_tr))
        best = es.result.xbest

        n_su = NUM_SERIAL_LAYERS * TRAINABLE_BLOCKS * NUM_SU_GATES * NUM_ROT_PARAMS
        weights_best = jnp.array(best[:n_su].reshape(
            NUM_SERIAL_LAYERS, TRAINABLE_BLOCKS, NUM_SU_GATES, NUM_ROT_PARAMS))
        alpha_best   = jnp.array(best[n_su:n_su+NUM_WIRES].reshape(1, NUM_WIRES))

        preds = [float(circuit_jit(alpha_best, weights_best, x_))
                 for x_ in x_te]
        r2 = r2_score(y_te, preds)
        print(f"  R²={r2:.4f}  alpha={alpha_best}")

        cmaes_results.append({
            "Approach": "CMA-ES", "Func": func, "Run": run,
            "R2_Score": r2, "final_alpha": alpha_best.tolist()
        })

cmaes_df = pd.DataFrame(cmaes_results)
cmaes_df.to_csv(OUT_DIR / "cmaes_results.csv", index=False)
print(f"\nCMA-ES done. Success rate (R²≥0.95): "
      f"{(cmaes_df.R2_Score>=0.95).mean()*100:.0f}%")

## Experiment 2: Random uniform initialization

In [ ]:
import optax

LR         = 0.001
MAX_STEPS  = 100 if SMOKE_TEST else 5000
BATCH_SIZE = 40

init_types = ["unary_perturbed", "random_uniform"]
random_results = []

for init_type in init_types:
    for func in range(NUM_FUNCTIONS):
        ds = datasets[func]
        x_tr, y_tr = ds["x_train"], ds["y_train"]
        x_te, y_te = ds["x_test"],  ds["y_test"]

        for run in range(NUM_RUNS):
            print(f"{init_type}  func={func+1}  run={run+1}/{NUM_RUNS}")

            if init_type == "unary_perturbed":
                alpha = jnp.array([[1.01, 1.02, 1.03]])
            else:
                alpha_vals = np.random.uniform(1.0, ALPHA_MAX, size=NUM_WIRES)
                alpha = jnp.array([alpha_vals])

            weights = jnp.array(weights_ones(
                NUM_SERIAL_LAYERS, TRAINABLE_BLOCKS,
                NUM_SU_GATES, NUM_ROT_PARAMS
            ))
            params    = {"weights_SU": weights, "alpha": alpha}
            opt       = optax.adam(LR)
            opt_state = opt.init(params)

            def cost_fn(p, x, y):
                preds = [circuit_jit(p["alpha"], p["weights_SU"], x_)
                         for x_ in x]
                return square_loss(y, preds).squeeze()

            @jax.jit
            def update(p, s, x, y):
                loss, g = jax.value_and_grad(cost_fn)(p, x, y)
                u, s2   = opt.update(g, s, p)
                return optax.apply_updates(p, u), s2, loss

            for step in range(MAX_STEPS):
                idx = np.random.randint(0, len(x_tr), BATCH_SIZE)
                params, opt_state, c = update(
                    params, opt_state,
                    jnp.array(x_tr[idx]), jnp.array(y_tr[idx])
                )

            preds = [float(circuit_jit(params["alpha"],
                                       params["weights_SU"], x_))
                     for x_ in x_te]
            r2 = r2_score(y_te, preds)
            print(f"  R²={r2:.4f}")

            random_results.append({
                "Approach":    init_type,
                "Func":        func,
                "Run":         run,
                "R2_Score":    r2,
                "final_alpha": params["alpha"].tolist()
            })

random_df = pd.DataFrame(random_results)
random_df.to_csv(OUT_DIR / "random_init_results.csv", index=False)
print("\nRandom init done.")
for it in init_types:
    sub = random_df[random_df.Approach == it]
    print(f"  {it}: success={( sub.R2_Score>=0.95).mean()*100:.0f}%  "
          f"median R²={sub.R2_Score.median():.3f}")

## Plots — fig:cmaes_analysis and baseline table

In [ ]:
from paper_style import boxplot_panel

# Load results (allows re-running plot cell independently)
cmaes_df  = pd.read_csv(OUT_DIR / "cmaes_results.csv")
random_df = pd.read_csv(OUT_DIR / "random_init_results.csv")

data = {
    r"Unary init":        random_df[random_df.Approach=="unary_perturbed"].R2_Score.values,
    r"Random init":       random_df[random_df.Approach=="random_uniform"].R2_Score.values,
    r"CMA-ES":            cmaes_df.R2_Score.values,
}
colors = [RED, ORANGE, BLUE]

fig, ax = plt.subplots(figsize=(3.5, 3.0))
boxplot_panel(ax, data, colors)
plt.tight_layout()
plt.savefig("cmaes_analysis.pdf", dpi=600, bbox_inches="tight")
plt.savefig("cmaes_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")

## Summary statistics

In [ ]:
print(f"{'Approach':<22} {'N':>4} {'Median':>7} {'Success%':>9}")
print("-" * 46)
for label, vals in data.items():
    print(f"{label:<22} {len(vals):>4} {np.median(vals):>7.3f} "
          f"{np.mean(vals>=0.95)*100:>8.0f}%")